<a href="https://colab.research.google.com/github/svallejovera/cta_updated/blob/main/Fine_Tuned_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Using Fine-Tuned Models

One limitation of Transformer-based models is that they require a fair amount of computational power. This can be especially challenging on older computers, though even newer machines may struggle with larger models. Fortunately, several cloud-computing platforms provide enough power for the kinds of tasks we will begin with in this course.

In this lesson, we will use Google Colab to run already fine-tuned models. You can browse thousands of publicly available fine-tuned models at [Hugging Face](https://huggingface.co).

### By the end of this notebook, you should be able to:

- explain what a pipeline is in the `transformers` library
- run a fine-tuned model for different NLP tasks
- interpret model outputs for masked language modeling, sentiment analysis, and NER
- apply a pipeline to many texts in a dataframe
- recognize that pre-trained and fine-tuned models still require evaluation and caution

> **Note:** The first time you run a pipeline, Colab may take a little while because it has to download the model and tokenizer files. This is normal.

## Pipelines

We can access many fine-tuned models through **pipelines**, which are ready-made interfaces in the `transformers` library. Pipelines let us use a model for a specific task without having to reproduce all the code that was originally used to train it.

Let’s begin with a masked-language model. We will use RoBERTa to predict which word best fits in a sentence containing a `<mask>` token.

In [ ]:
from transformers import pipeline

# We define a pipeline for the "fill-mask" task using the RoBERTa base model.
unmasker = pipeline("fill-mask", model="FacebookAI/roberta-base")

# The model will predict which word best completes the sentence.
# The first time you run this, Colab will download the tokenizer and model files.
unmasker("Tiago Ventura is the <mask> teacher of all Georgetown.")

The results are often surprisingly plausible, though not always correct. Let’s try a few more examples and see how the model behaves with different languages and contexts.

In [ ]:
requests_from_class = [
    "This class was really <mask>.",
    "The cat is hungry so it <mask>.",
    "La casa es grande y tiene muchos <mask>.",
    "Recife é uma cidade em <mask>."
]

for request in requests_from_class:
    print(request)
    print(unmasker(request))
    print()

Before we fine-tune our own models, let’s try a model that has already been fine-tuned for a specific task. We will use a multilingual DistilBERT model for sentiment analysis.

DistilBERT is a smaller, faster version of BERT that retains much of BERT’s performance while being more efficient to run.

In [ ]:
from transformers import pipeline

multilingual_sentiment = pipeline(
    "text-classification",
    model="lxyuan/distilbert-base-multilingual-cased-sentiments-student",
    top_k=None
)

examples = [
    "This class is the greatest waste of time.",
    "Essa palestra é uma completa perda de tempo.",
    "Esta clase no está ni tan bien, ni tan mal.",
    "This class is not good, but it is not bad either.",
    "Gosto muito de te ver, leãozinho. Caminhando sob o sol. Gosto muito de você, leãozinho."
]

for text in examples:
    print(text)
    print(multilingual_sentiment(text))
    print()

How should we read these results?

The model returns a score for each sentiment category. These scores behave like probabilities: higher values indicate greater confidence in that category. Notice that the same meaning expressed in different languages may not receive exactly the same scores. This can happen because the model may have seen different amounts or types of training data in different languages, and because translation rarely preserves tone perfectly.

A useful thing to notice is that pipelines often return their results as a list of dictionaries. That format is flexible, but it usually needs some cleaning before we can analyze the results in a dataframe. This is very common in real text-analysis workflows.

In [ ]:
import pandas as pd
from tqdm import tqdm
from transformers import pipeline

# Load a sample dataset of texts
ex_text = pd.read_csv(
    "https://raw.githubusercontent.com/svallejovera/iesp-uerj/main/sample_sentiment.csv",
    on_bad_lines="skip"
)

ex_text.head()

In [ ]:
# Load a sentiment analysis pipeline
model_path = "cardiffnlp/twitter-roberta-base-sentiment-latest"
sa_pipeline = pipeline("sentiment-analysis", model=model_path, tokenizer=model_path)

# Apply the model to each text
predicted_text = []

for text in tqdm(ex_text["text"].fillna("")):
    prediction = sa_pipeline(text)[0]
    predicted_text.append(prediction)

# Store predictions in separate columns
ex_text["label"] = [pred["label"] for pred in predicted_text]
ex_text["score"] = [pred["score"] for pred in predicted_text]

ex_text.head()

Let’s now try a model fine-tuned for **Named Entity Recognition (NER)**. NER models identify spans of text that refer to entities such as people, organizations, locations, and countries.

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

tokenizer = AutoTokenizer.from_pretrained("dslim/bert-base-NER")
model = AutoModelForTokenClassification.from_pretrained("dslim/bert-base-NER")

ner_pipeline = pipeline("ner", model=model, tokenizer=tokenizer)

example = (
    "Tiago Ventura is a professor at Georgetown University, "
    "which is in the United States. Yet, he was born in Belém, "
    "which is a city in a state in Brazil."
)

ner_results = ner_pipeline(example)
ner_results

In [ ]:
ner_pipeline_grouped = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)

grouped_results = ner_pipeline_grouped(example)
grouped_results

Why does the first output sometimes split words into pieces?

Many Transformer models use **subword tokenization**, meaning that a word may be broken into smaller units before being processed. This is useful because it helps the model handle rare words, unfamiliar words, and morphological variation. By using `aggregation_strategy="simple"`, we can combine many of these subword pieces back into more readable entity predictions.

### Try it yourself

Now you can test the behavior yourself. Change one of the input sentences above and see how the model responds. Try:
- a very short sentence
- a sarcastic sentence
- a sentence in a language other than English
- a sentence with ambiguous sentiment

You can now explore many other fine-tuned models on Hugging Face. This is one of the major advantages of the current NLP ecosystem: you often do not need to train a model from scratch in order to begin experimenting.

At the same time, you should be cautious. Not all fine-tuned models are equally good, and many will have been trained on data that differ from your own texts. Sometimes that mismatch will not matter much; sometimes it will matter a great deal. Choosing a model is therefore not just a technical decision, but also a substantive one.

### A reminder about limitations

These models can produce fluent and plausible outputs, but plausible does not always mean correct. Their performance depends on the data they were trained on, the task they were fine-tuned for, and how similar your own texts are to that training data.